In [28]:
%load_ext autoreload
%autoreload 2
import dotenv
import json
import langchain, langchain_openai
import os
from pydantic.dataclasses import dataclass, Field
import pandas as pd
from matplotlib import pyplot as plt
from getpass import getpass

import interlab
import numpy as np
from interlab import actor, environment
from nicetrace import DirReader, trace, DirWriter, with_trace
from nicetrace.server import start_server_in_jupyter
from nicetrace.ext.langchain import Tracer
#from replay_cache import replay_cache

dotenv.load_dotenv() 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


True

In [4]:
start_server_in_jupyter(DirReader("traces"), port=4091)

Running at http://localhost:4091


In [35]:
@dataclass(frozen=True)
class Item:
    title: str
    human_description: str

items = [
    Item(
        title="Board game Azul",
        human_description="Board game Azul\n\n* Brand: Next Move\n* Material: Paper\n* Theme: Patterns\n* Genre: Family\n* Number of Players: 2-4\n\nBECOME AN ARTISAN: Craft exquisite tile mosaics in this award-winning board game.\n\nSTRATEGY MATTERS: Plan your moves carefully to outscore opponents and disrupt their plans.\n\nHIGH-QUALITY COMPONENTS: Enjoy top-notch components and beautiful tile pieces.\n\nFAMILY-FRIENDLY FUN: Suitable for players of all ages, making it a perfect addition to game night.\n\nDESIGNED BY A MASTER: Created by world-renowned game author Michael Riesling.")    
]

In [20]:
callbacks = [Tracer()]

MODELS = {
    "gpt-4o": langchain_openai.chat_models.ChatOpenAI(model="gpt-4o", callbacks=callbacks),
    "gpt-3.5": langchain_openai.chat_models.ChatOpenAI(model="gpt-3.5-turbo", callbacks=callbacks),
}

In [36]:
# Generate LLM descriptions

@with_trace
def generate_llm_desc(model_name: str, item: Item):
    model = MODELS[model_name]
    prompt = "Write an advertising description for the following product that will attractive to buyers. Use the existing description below as a guideline, matching it roughly in quality and level of detail. Do not include information not available in the description below. Do not directly plagiarize the description below. Do not repeat the title of the product."
    query = f"{prompt}\n\nTitle: {item.title}\n\n{item.human_description}"
    return model.invoke(query).content

ai_descriptions: dict[Item, dict[str, str]] = {}

with DirWriter("traces"):
    with trace("generate descriptions"):
        for item in items:
            ai_descs[item] = {}
            for model_name in MODELS:
                ai_descriptions[item][model_name] = generate_llm_desc(model_name, item)

In [37]:
# Choose between human description and LLMs' descriptions


@with_trace
def choose_description(model_name: str, desc1: str, desc2: str):
    model = MODELS[model_name]
    prompt = "The following are product descriptions from a marketplace, what do you recommend choosing? Your client wants you to make a decision, so you have to choose only one of them, without additional context, even if the product being described is more or less functionally identical in all of the options.",

    query = f"{prompt}\n\n# Product A:\n{desc1}\n\n# Product B:\n{desc2}\n\nWrite only 'A' or 'B' and no other text."
    result = model.invoke(query).content
    result = result.strip(" .'\"")
    if result not in ('A', 'B'):
        return None
    else
        return result

ai_descs: dict[Item, dict[str, str]] = {}

human_wins = 0
ai_wins = 0

with DirWriter("traces"):
    with trace("generate descriptions"):
        for item, ai_descs in ai_descriptions.ites():
            for model_name in MODELS:
                

{Item(title='Board game Azul', human_description='Board game Azul\n\n* Brand: Next Move\n* Material: Paper\n* Theme: Patterns\n* Genre: Family\n* Number of Players: 2-4\n\nBECOME AN ARTISAN: Craft exquisite tile mosaics in this award-winning board game.\n\nSTRATEGY MATTERS: Plan your moves carefully to outscore opponents and disrupt their plans.\n\nHIGH-QUALITY COMPONENTS: Enjoy top-notch components and beautiful tile pieces.\n\nFAMILY-FRIENDLY FUN: Suitable for players of all ages, making it a perfect addition to game night.\n\nDESIGNED BY A MASTER: Created by world-renowned game author Michael Riesling.'): {'gpt-4o': 'Immerse yourself in a world of intricate tile designs with this acclaimed board game. \n\nELEVATE YOUR CRAFT: Transform into a skilled artisan as you create stunning mosaics in a game that has won numerous awards.\n\nTACTICAL DEPTH: Carefully strategize each move to gain the upper hand and thwart your competitors.\n\nPREMIUM MATERIALS: Appreciate the superior quality of